In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
import torch_geometric.transforms as T
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [3]:
# Define the GAT model
# this implementation is credit to pytorch_geometric examples
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, data):
        h, edge_index = data.x, data.edge_index

        h = F.dropout(h, p=0.6, training=self.training)
        h = F.elu(self.conv1(h, edge_index))
        h = F.dropout(h, p=0.6, training=self.training)
        h = self.conv2(h, edge_index)

        return h

# Load the datasets
citeseer_dataset = Planetoid(root='/tmp/Citeseer', name='Citeseer', transform=T.NormalizeFeatures())
data = citeseer_dataset[0]
data = data.to(device)

Processing...
Done!


In [4]:
h_channels = 64
heads = 8
model = GAT(citeseer_dataset.num_features, h_channels, citeseer_dataset.num_classes, heads)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
def train(model, data, train_mask, labels):
    model.train()

    optimizer.zero_grad()
    logits = model(data.cuda())
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [5]:
train(model, data, data.train_mask, data.y)

1.7930103540420532

In [6]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)

    acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
    return acc

for epoch in range(0, 200):
    loss = train(model, data, data.train_mask, data.y)
    acc = test()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 1.7833, Accuracy: 0.4450
Epoch: 001, Loss: 1.7641, Accuracy: 0.3740
Epoch: 002, Loss: 1.7538, Accuracy: 0.4740
Epoch: 003, Loss: 1.7406, Accuracy: 0.5140
Epoch: 004, Loss: 1.7231, Accuracy: 0.4740
Epoch: 005, Loss: 1.7031, Accuracy: 0.4720
Epoch: 006, Loss: 1.6867, Accuracy: 0.5250
Epoch: 007, Loss: 1.6929, Accuracy: 0.5510
Epoch: 008, Loss: 1.6540, Accuracy: 0.4730
Epoch: 009, Loss: 1.6491, Accuracy: 0.4120
Epoch: 010, Loss: 1.6130, Accuracy: 0.4130
Epoch: 011, Loss: 1.5826, Accuracy: 0.4770
Epoch: 012, Loss: 1.5771, Accuracy: 0.5480
Epoch: 013, Loss: 1.5379, Accuracy: 0.6590
Epoch: 014, Loss: 1.5593, Accuracy: 0.7060
Epoch: 015, Loss: 1.5001, Accuracy: 0.7180
Epoch: 016, Loss: 1.4298, Accuracy: 0.7010
Epoch: 017, Loss: 1.4384, Accuracy: 0.6770
Epoch: 018, Loss: 1.4444, Accuracy: 0.6660
Epoch: 019, Loss: 1.3690, Accuracy: 0.6630
Epoch: 020, Loss: 1.3911, Accuracy: 0.6660
Epoch: 021, Loss: 1.3275, Accuracy: 0.6750
Epoch: 022, Loss: 1.3083, Accuracy: 0.6720
Epoch: 023,

In [7]:
torch.save(model.state_dict(), 'citeseer_gat.pt')